In [5]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
from PIL import Image

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using {device} device')

Using cuda device


In [23]:
# Training model

class Net(nn.Module):
  def __init__(self):
    super().__init__()
    self.flatten = nn.Flatten()
    self.linear_relu_stack = nn.Sequential(
        nn.Linear(28*28, 512),
        nn.ReLU(),
        nn.Linear(512,512),
        nn.ReLU(),
        nn.Linear(512,10),
    )

  def forward(self, x):
    x = self.flatten(x)
    logits = self.linear_relu_stack(x)
    return logits

In [26]:
# Training function

def train(dataloader, model, loss_fn, optimizer):
 
  size = len(dataloader.dataset)
  model.train()

  for batch, (X, y) in enumerate(dataloader):
    X, y = X.to(device), y.to(device)

    pred = model(X)
    loss = loss_fn(pred, y)

    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    if batch % 100 == 0:
      loss, current = loss.item(), batch * len(X)
      print(f"loss: {loss:>7} [{current:>5d}/{size:>5d}]")


In [18]:
# Test Function
def test(dataloader, model, loss_fn):
  
  size = len(dataloader.dataset)
  num_batches = len(dataloader)
  model.eval()
  test_loss = 0.0
  correct = 0.0

  with torch.inference_mode():
    for X, y in dataloader:
      X, y = X.to(device), y.to(device)
      pred = model(X)
      test_loss = loss_fn(pred, y).item()
      correct += (pred.argmax(1) == y).type(torch.float).sum().item()
  test_loss /= num_batches
  correct /= size

  print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg. loss:{test_loss:>8f} \n")

In [27]:
def run_and_save_training(save_path="mnist_fashion_model.pth"):
    
    training_data = datasets.FashionMNIST('./data', 
                               train=True, 
                               download=True, 
                               transform=transforms.ToTensor()
                               )

    test_data = datasets.FashionMNIST('./data', 
                            train=False, 
                            download=True,
                            transform=transforms.ToTensor()
                            )

    batch_size = 64

    training_dataloader = DataLoader(training_data, batch_size=batch_size, shuffle=True)
    test_dataloader = DataLoader(test_data, batch_size=batch_size, shuffle=False)
    
    epochs = 5
    model = Net().to(device)
    loss_fn = nn.CrossEntropyLoss() # Loss function
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    for t  in range(epochs):
        print(f"Epoch {t+1}============================================")
        train(training_dataloader, model, loss_fn, optimizer)
        test(test_dataloader, model, loss_fn)
    
    print("Done!")

    torch.save(model.state_dict(), save_path)
    print(f"Saved PyTorch Model State to: {save_path}")

In [28]:
run_and_save_training()

Epoch 1============================================
loss: 2.3028993606567383 [    0/60000]
loss: 0.522159218788147 [ 6400/60000]
loss: 0.4922235310077667 [12800/60000]
loss: 0.49248528480529785 [19200/60000]
loss: 0.46517542004585266 [25600/60000]
loss: 0.7573828101158142 [32000/60000]
loss: 0.27456459403038025 [38400/60000]
loss: 0.4597271680831909 [44800/60000]
loss: 0.4430626928806305 [51200/60000]
loss: 0.3757781386375427 [57600/60000]
Test Error: 
 Accuracy: 85.5%, Avg. loss:0.001564 

Epoch 2============================================
loss: 0.2969391644001007 [    0/60000]
loss: 0.4255847632884979 [ 6400/60000]
loss: 0.43976378440856934 [12800/60000]
loss: 0.22757519781589508 [19200/60000]
loss: 0.27869510650634766 [25600/60000]
loss: 0.42900550365448 [32000/60000]
loss: 0.28522706031799316 [38400/60000]
loss: 0.3860560953617096 [44800/60000]
loss: 0.1982102394104004 [51200/60000]
loss: 0.3534456491470337 [57600/60000]
Test Error: 
 Accuracy: 85.9%, Avg. loss:0.001958 

Epoch 3=